In [ ]:
import matplotlib.pylab as plt
import matplotlib as mpl
import xarray as xr
import pint_xarray
import numpy as np
import cftime
from functools import partial

from pism_terra.processing import integrate_rate, preprocess_netcdf

ref_year = "1990"

In [ ]:
ds_free = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_free//output/scalar/basin_g1500m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
ds_free = ds_free.expand_dims({"uq_id": ["free"]})
ds_prescribed = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_1985_prescribed/output/scalar/basin_g1500m_id_CESM2-WACCM_historical_free_1985-01-01_2015-01-01.nc")
ds_prescribed = ds_prescribed.expand_dims({"uq_id": ["prescribed"]})
ds_inv = xr.open_dataset("/Users/andy/base/pism-terra/2026_08_ismip7_hist_2007_inv_func/output/scalar/basin_g1500m_id_CESM2-WACCM_uq_1_historical_free_2007-01-01_2015-01-01.nc")
ds_inv = ds_inv.expand_dims({"uq_id": ["inv"]})
# ds_s = ds_s.expand_dims({"uq_id": ["all"]}).convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time')
#ds = xr.merge([ds_free, ds_prescribed, ds_inv], compat="no_conflicts", join="outer")
ds = xr.merge([ds_free, ds_prescribed], compat="no_conflicts", join="outer")
ds = ds.sel(basin="GIS")

In [ ]:
ds = ds.convert_calendar("standard", use_cftime=False).resample(time='MS').mean('time').pint.quantify()

In [ ]:
grace = xr.open_dataset("/Users/andy/base/pism-ragis/data/grace/greenland_mass_balance.nc").squeeze().pint.quantify()
mankoff = xr.open_dataset("/Users/andy/base/pism-ragis/data/mass_balance/mankoff_greenland_mass_balance_clean.nc").pint.quantify()
mankoff = mankoff.sum(dim="region").resample(time='MS').mean("time").pint.to("Gt/yr")

sigma = 2.0
mankoff_cmb = integrate_rate(mankoff.MB)
mankoff_cmb = (mankoff_cmb - mankoff_cmb.sel(time=ref_year, method="nearest"))
mankoff_mb = mankoff.MB.resample(time='YS').mean("time")
mankoff_mb_err = mankoff.MB_err.resample(time='YS').mean("time")
mankoff_smb = mankoff.SMB.resample(time='YS').mean("time")
mankoff_smb_err = mankoff.SMB_err.resample(time='YS').mean("time")
mankoff_glf = -mankoff.D.resample(time='YS').mean("time")
mankoff_glf_err = -mankoff.D_err.resample(time='YS').mean("time")


In [ ]:
mass = ds.ice_mass_glacierized

d = ds.grounding_line_flux_nonneg * xr.DataArray(1500).pint.quantify("m") ** 2

mass = integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux) + integrate_rate(ds.tendency_of_ice_mass_due_to_discharge)
mass = integrate_rate(ds.tendency_of_ice_mass_due_to_surface_mass_flux) + integrate_rate(d)

mass = mass - mass.sel(time=ref_year, method="nearest")
mass = mass.pint.to("Gt")
#mass = (ds.tendency_of_ice_mass.pint.to("Gt/yr") - xr.DataArray(400).pint.quantify("Gt/yr")).cumsum(dim="time") 
#mass = mass - mass.sel(time="2002", method="nearest")

fig, ax = plt.subplots(1, 1)
mass.plot(hue="uq_id", ax=ax, lw=1, add_legend=True)
#grace.cumulative_mass_balance.plot(ax=ax, color="#DC267F")
mankoff_cmb.plot(ax=ax, lw=2, color="0.5")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))


In [ ]:
glf = ds.tendency_of_ice_mass_due_to_discharge
#glf = d.pint.to("Gt/yr")
smb = ds.tendency_of_ice_mass_due_to_surface_mass_flux
mb = smb + glf 

rc_params = {
    "font.size": 6,
        # Add other rcParams settings if needed
}

with mpl.rc_context(rc=rc_params):

    fig, axs = plt.subplots(3, 1, sharex=True, figsize=(4.8, 4.4))
    axs[0].fill_between(mankoff_mb.time, mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
    axs[1].fill_between(mankoff_mb.time, mankoff_smb - sigma * mankoff_smb_err, mankoff_smb + sigma * mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
    axs[2].fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)
    
    mankoff_mb.resample(time='YS').mean('time').plot(ax=axs[0], color="0.5", lw=2)
    mankoff_smb.resample(time='YS').mean('time').plot(ax=axs[1], color="0.5", lw=2)
    mankoff_glf.resample(time='YS').mean('time').plot(ax=axs[2], color="0.5", lw=2)
    for k, (da, ls) in enumerate([(mb, "solid"), (smb, "dotted"), (glf, "dashed")]):
        ax = axs[k]
        # for u in da["uq_id"].values:
        #     #da.sel(uq=u).plot(ax=ax, color=palette[u], ls=ls)
        #     da.sel(uq_id=u).plot(ax=ax, color=palette[u], ls="solid", lw=2)
        da.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, add_legend=False if k > 0 else True)        
        ax.set_title(None)
        ax.set_xlabel(None)
        ax.axhline(0, color="k", lw=0.5, ls="dotted")
        ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
        #ax.set_ylim(-1000, 1000)
    axs[-1].set_xlabel("Year")
    fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6.4, 6.2))
ax.fill_between(mankoff_mb.time, mankoff_mb - sigma * mankoff_mb_err, mankoff_mb + sigma * mankoff_mb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_smb - sigma * mankoff_smb_err, mankoff_smb + sigma * mankoff_smb_err, color="0.5", alpha=0.25, lw=0)
ax.fill_between(mankoff_mb.time, mankoff_glf - sigma * mankoff_glf_err, mankoff_glf + sigma * mankoff_glf_err, color="0.5", alpha=0.25, lw=0)

mankoff_mb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="solid")
mankoff_smb.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dashed")
mankoff_glf.resample(time='YS').mean('time').plot(ax=ax, color="0.5", lw=2, ls="dotted")

mb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="solid", add_legend=True)        
ax.set_prop_cycle(None)
smb.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="dashed", add_legend=False)        
ax.set_prop_cycle(None)
glf.compute().resample(time='YS').mean('time').plot(hue="uq_id", ax=ax, lw=1.5, ls="dotted", add_legend=False)     
ax.set_prop_cycle(None)


ax.set_title(None)
ax.axhline(0, color="k", lw=0.5, ls="dotted")
ax.set_xlim(np.datetime64("1985"), np.datetime64("2015"))
ax.set_ylim(-800, 800)

In [ ]:
ds.tendency_of_ice_mass_due_to_surface_mass_flux